In [1]:
import google.auth
import numpy as np
import pandas as pd
import pygris 
import geopandas as gpd
from calitp_data_analysis import geography_utils
from calitp_data_analysis.sql import to_snakecase
from shared_utils import arcgis_query

In [2]:
from calitp_data_analysis import get_fs
fs = get_fs()

In [3]:

import os
from typing import List, Optional, Union
import pyarrow.dataset as ds
from google.cloud import storage

In [4]:
import google.auth
import pandas_gbq

credentials, project = google.auth.default()
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas

In [5]:


@cache
def gcs_geopandas():
    return GCSGeoPandas()

In [6]:
@cache
def gcs_pandas():
    return GCSPandas()

In [7]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Load Data
### Census Block
STATEFP20

2‑digit FIPS code for the state
Example: 06 = California

COUNTYFP20

3‑digit FIPS code for the county
Example: 001 = Alameda County

TRACTCE20

Census tract code (6 digits, no decimal)

BLOCKCE20

Census block code (4 digits)

GEOID20

Full 15‑digit concatenated identifier:
STATEFP + COUNTYFP + TRACTCE + BLOCKCE

Example: 060014001001234

NAME20

Block name (usually same as block number)

MTFCC20

Feature class code indicating type of Census block
For blocks this is usually:

G5030 = Census Block



UR20

Urban/rural classification flag

U = Urban
R = Rural



UACE20

Urban Area Census Code
If the block is inside an urban area, this is the UA code
Example: 06004 for Oakland UA

UATYPE20

Urban Area Type

U = Urbanized area (50,000+ people)
C = Urban cluster (2,500–49,999 people)



FUNCSTAT20

Functional status of the block
For blocks:

S = Statistical entity (standard for census blocks)



ALAND20

Land area in square meters

AWATER20

Water area in square meters

INTPTLAT20

Internal point latitude (label point for the block)

INTPTLON20

Internal point longitude

HOUSING20

Number of housing units in the block (2020 Census)

In [8]:
def chunked(seq, size):
    """Yield successive chunks of length 'size' from a sequence."""
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [9]:
def load_ca_blocks(year:int, variables:str, county_code: str):
    """
    Load all census blocks for California (2020 PL94 decennial).
    Downloads one county at a time because the API does not allow
    state-level block requests.
    """
    df = pygris.blocks(
        state="06",     # California
        county=county_code,   
        year=2020,
        cache=True
       )

    # Reproject
    df = df.to_crs(geography_utils.CA_NAD83Albers_ft)

    # Buffer
    df["b250"] = df.buffer(250)
    df = df.drop(columns = ["geometry"])

    # Save locally first
    df.to_parquet("./chunked.parquet")

    # Save to GCS
    ca_counties = to_snakecase(pygris.counties(state="CA", year=year)[["NAME","COUNTYFP"]])
    file_name = ca_counties.loc[ca_counties.countyfp == county_code].name.iloc[0]
    df["county_name"] = file_name
    
    # Put the local file into the GCS bucket
    df.to_parquet("./chunked.parquet")
    fs.put("./chunked.parquet", f"gs://calitp-analytics-data/data-analyses/equity_index/census_blocks/census_blocks_county_{file_name}.parquet")
    return df

In [10]:
#for county in ca_counties_list:
#    block_gdf = load_ca_blocks(2020, "P1_001N", county)
#    print(f"Done with {county}")

In [11]:
def _parse_gcs_path(gcs_path: str):
    """
    Split a GCS URL into (bucket, prefix) without leading 'gs://'.
    """
    if not gcs_path.startswith("gs://"):
        raise ValueError(f"Expected a 'gs://' path, got: {gcs_path}")
    no_scheme = gcs_path[5:]
    bucket, *rest = no_scheme.split("/", 1)
    prefix = rest[0] if rest else ""
    if prefix and not prefix.endswith("/"):
        prefix += "/"
    return bucket, prefix

In [12]:
def list_gcs_files(gcs_folder: str, extensions: Optional[List[str]] = None) -> list:
    """
    List all files in a GCS 'folder' (prefix). Optionally filter by extensions.
    Returns full 'gs://...' URIs.
    """
    bucket_name, prefix = _parse_gcs_path(gcs_folder)
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    uris: List[str] = []
    for blob in client.list_blobs(bucket_name, prefix=prefix):
        # Skip "directory placeholders"
        name = blob.name
        if name.endswith("/"):
            continue
        if extensions:
            if not any(name.lower().endswith(ext.lower()) for ext in extensions):
                continue
        uris.append(f"gs://{bucket_name}/{name}")

    return sorted(uris)

In [13]:

try:
    import geopandas as gpd
    _HAS_GPD = True
except Exception:
    _HAS_GPD = False


In [14]:
def concat_gcs_folder(
    gcs_folder: str = "gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset: bool = True,
    file_types: Optional[List[str]] = None,
    geometry: bool = False,
    dtype_overrides: Optional[dict] = None,
    use_threads: bool = True,
) -> Union[pd.DataFrame, "gpd.GeoDataFrame"]:
    """
    Concatenate all files in a GCS folder into a single DataFrame.
    """

    # Default supported formats
    if file_types is None:
        file_types = ["parquet", "csv", "feather", "geojson", "json"]

    files = list_gcs_files(gcs_folder, extensions=[f".{ext}" for ext in file_types])

    if not files:
        raise FileNotFoundError(f"No files found under: {gcs_folder} with types {file_types}")

    # Disable arrow.dataset for parquet — because GCS credentials fail there
    all_parquet = all(f.lower().endswith(".parquet") for f in files)
    
    # Updated behavior: Always use gcs_geopandas().read_parquet for parquet files
    if all_parquet:
        frames = []
        for uri in files:
            frames.append(gcs_geopandas().read_parquet(uri))
        df = pd.concat(frames, ignore_index=True)

        if dtype_overrides:
            df = df.astype(dtype_overrides, errors="ignore")
        return df

    # Otherwise fall back to file-by-file reading
    frames: List[Union[pd.DataFrame, "gpd.GeoDataFrame"]] = []

    for uri in files:
        lower = uri.lower()

        if lower.endswith(".parquet"):
            frames.append(gcs_geopandas().read_parquet(uri))

        elif lower.endswith(".feather"):
            frames.append(pd.read_feather(uri))

        elif lower.endswith(".csv"):
            frames.append(pd.read_csv(uri, low_memory=False))

        elif lower.endswith(".geojson") or (lower.endswith(".json") and "geo" in os.path.basename(uri).lower()):
            if not _HAS_GPD:
                raise ImportError("geopandas not installed—install it or set geometry=False.")
            gdf = gpd.read_file(uri)
            frames.append(gdf)

        else:
            print(f"[concat_gcs_folder] Skipping unsupported file: {uri}")

    if not frames:
        raise FileNotFoundError(f"Found files, but none were readable with the allowed types: {file_types}")

    # If any frames are GeoDataFrames, concatenate as GeoDataFrame
    if _HAS_GPD and any(isinstance(f, gpd.GeoDataFrame) for f in frames):
        df = pd.concat(frames, ignore_index=True)
        if "geometry" in df.columns and not isinstance(df, gpd.GeoDataFrame):
            df = gpd.GeoDataFrame(df, geometry="geometry", crs=frames[0].crs if hasattr(frames[0], "crs") else None)
    else:
        df = pd.concat(frames, ignore_index=True)

    if dtype_overrides:
        df = df.astype(dtype_overrides, errors="ignore")

    df = to_snakecase(df)
    return df


In [15]:
"""
census_blocks_gdf = concat_gcs_folder(
    gcs_folder = "gs://calitp-analytics-data/data-analyses/equity_index/census_blocks",
    prefer_arrow_dataset =True,
    file_types = None,
    geometry = True,
    dtype_overrides = None,
    use_threads = True,
)
"""

In [23]:
census_blocks = gcs_geopandas().read_parquet("gs://calitp-analytics-data/data-analyses/shared_data/census_blocks_combined.parquet")

### Public Road Functional Classification
**Amanda** Need to fix: URL maxes out at 2000 rows when there are thousands more. 

In [24]:
# https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson

In [25]:
public_road_url = "https://caltrans-gis.dot.ca.gov/arcgis/rest/services/CHhighway/CRS_Functional_Classification/FeatureServer/0/"

In [26]:
public_road_gdf = to_snakecase(gcs_geopandas().read_parquet("gs://calitp-analytics-data/data-analyses/shared_data/public_road_functional_classification.parquet"))

In [27]:
public_road_gdf.f_system.unique()

array([4, 3, 5, 2, 7, 6, 1], dtype=int32)

In [28]:
interstate_freeway = public_road_gdf.loc[public_road_gdf["f_system"].isin([1,2])]

In [29]:
len(interstate_freeway), len(public_road_gdf)

(15895, 779834)

In [30]:
interstate_freeway = interstate_freeway.to_crs(geography_utils.CA_NAD83Albers_ft)

In [31]:
interstate_freeway["b50"] = interstate_freeway.geometry.buffer(50)

In [32]:
interstate_freeway = interstate_freeway.drop(columns = ["geometry"])

In [33]:
interstate_freeway = interstate_freeway.set_geometry("b50")

### TIMS Data

In [34]:
"""
tims_gdf = concat_gcs_folder(
    gcs_folder="gs://calitp-analytics-data/data-analyses/equity_index/tims/",
    prefer_arrow_dataset=True
)

tims_gdf = gpd.GeoDataFrame(
    tims_gdf, geometry=gpd.points_from_xy(tims_gdf.point_x, tims_gdf.point_y), crs=geography_utils.WGS84 
).to_crs(geography_utils.CA_NAD83Albers_ft)
"""

## Overlay TIMS with Public Road Functional Classification data for crashes we don't want. 
* Filter them out

In [92]:
tims_public_road = (
        tims_gdf.sjoin(interstate_freeway, how="inner", predicate="intersects")
        .reset_index(drop=True)
        .drop(columns=["index_right"])
    )

In [93]:
crashes_to_delete = list(tims_public_road.case_id.unique())

In [94]:
tims_gdf2 = tims_gdf.loc[~tims_gdf.case_id.isin(crashes_to_delete)]

In [96]:
tims_gdf2.collision_severity.unique()

array([4, 3, 2, 1])

In [105]:
tims_gdf2 = tims_gdf.loc[tims_gdf.collision_severity.isin([1,2])][["case_id",
                                                                  "accident_year",
                                                                  "geometry",
                                                                  "collision_severity"]]

In [104]:
tims_gdf2.columns

Index(['case_id', 'accident_year', 'proc_date', 'juris', 'collision_date',
       'collision_time', 'officer_id', 'reporting_district', 'day_of_week',
       'chp_shift', 'population', 'cnty_city_loc', 'special_cond', 'beat_type',
       'chp_beat_type', 'city_division_lapd', 'chp_beat_class', 'beat_number',
       'primary_rd', 'secondary_rd', 'distance', 'direction', 'intersection',
       'weather_1', 'weather_2', 'state_hwy_ind', 'caltrans_county',
       'caltrans_district', 'state_route', 'route_suffix', 'postmile_prefix',
       'postmile', 'location_type', 'ramp_intersection', 'side_of_hwy',
       'tow_away', 'collision_severity', 'number_killed', 'number_injured',
       'party_count', 'primary_coll_factor', 'pcf_code_of_viol',
       'pcf_viol_category', 'pcf_violation', 'pcf_viol_subsection',
       'hit_and_run', 'type_of_collision', 'mviw', 'ped_action',
       'road_surface', 'road_cond_1', 'road_cond_2', 'lighting',
       'control_device', 'chp_road_type', 'pedestrian_

In [106]:
tims_gdf2.to_parquet("./tims_relevant_crashes_2026.parquet")
fs.put("./tims_relevant_crashes_2026.parquet", "gs://calitp-analytics-data/data-analyses/equity_index/analysis_2026/tims_relevant_crashes_2026.parquet")

[None]

In [107]:
tims_gdf2.shape

(57286, 4)

# Overlay crashes we are interested in with Census Block

In [108]:
crashes_m1 = (
        tims_gdf2.sjoin(census_blocks_gdf, how="left", predicate="within")
        .reset_index(drop=True)
        .drop(columns=["index_right"])
    )

In [109]:
census_blocks_gdf.shape

(519723, 19)

In [110]:
crashes_m1.shape

(236892, 22)

In [111]:
crashes_m1.case_id.nunique()

57286

In [112]:
crashes_m1.case_id.value_counts().head()

case_id
9274064     18
91676653    17
91876693    16
9327460     16
9550520     16
Name: count, dtype: int64

In [113]:
crashes_m1.case_id.value_counts().describe()

count   57286.00
mean        4.14
std         1.91
min         1.00
25%         3.00
50%         4.00
75%         5.00
max        18.00
Name: count, dtype: float64

In [153]:
crashes_census_block = crashes_m1.groupby(["GEOID20"]).agg({"case_id":"nunique"}).reset_index().rename(columns = {"case_id":"n_crashes"})

In [154]:
# Find area of census blocks
census_blocks_gdf["census_block_area_sq_miles"] = (census_blocks_gdf.b250.area)/ 2_589_988.11

In [155]:
crashes_census_block2 = pd.merge(crashes_census_block, 
                                census_blocks_gdf[["GEOID20", "census_block_area_sq_miles"]],
                                on = "GEOID20",
                                how = "inner")

In [156]:
crashes_census_block2.shape

(122083, 3)

In [159]:
crashes_census_block2["crash_density"] = crashes_census_block2["n_crashes"] / crashes_census_block2["census_block_area_sq_miles"]

In [161]:

crashes_census_block2["crash_percentile_100"] = (
    crashes_census_block2["crash_density"].rank(pct=True) * 100
)


In [166]:
crashes_census_block2.sort_values(by = ["crash_percentile_100"], ascending = False).head(30)

,GEOID20,n_crashes,census_block_area_sq_miles,crash_density,crash_percentile_100
30533,060372382002019,8,0.13,61.97,100.00
7702,060133240023021,9,0.16,56.45,100.00
86190,060730025011011,7,0.13,54.48,100.00
30812,060372400101000,10,0.19,53.30,100.00
93359,060750129023007,11,0.22,50.18,100.00
86764,060730041013011,7,0.15,47.41,100.00
93276,060750124061000,11,0.23,47.11,100.00
93358,060750129023006,11,0.23,47.09,99.99
31320,060372412024010,10,0.21,46.84,99.99
93278,060750124061002,11,0.24,46.57,99.99
